# AI Feynman 2.0: データから物理法則を自動発見する

---

## 概要

このノートブックでは、**AI Feynman 2.0** を使って、数値データから背後にある数学的法則（数式）を自動的に発見する方法を学びます。

### AI Feynman とは？

AI Feynman は、MIT の Max Tegmark 教授らのチームが開発した**記号回帰（Symbolic Regression）**システムです。従来の機械学習とは異なり、ブラックボックスモデルではなく、人間が理解できる数式を出力します。

### 主な特徴

- **物理学の知見を活用**: 対称性、分離可能性などの物理的性質を利用して効率的に探索
- **パレート最適化**: 複雑さと精度のトレードオフを考慮した複数の候補を提示
- **ハイブリッドアプローチ**: 総当たり探索 + ニューラルネットワーク + 多項式フィッティング

### 参考論文

1. Udrescu & Tegmark, "AI Feynman: A Physics-Inspired Method for Symbolic Regression" (Science Advances, 2020)
2. Udrescu et al., "AI Feynman 2.0: Pareto-optimal symbolic regression exploiting graph modularity" (arXiv:2006.10782, 2020)

---

## 1. 環境セットアップ

まず、必要なパッケージをインストールします。

### 注意事項
- インストール後、**ランタイムの再起動が必要**な場合があります
- Fortran コンパイラ（gfortran）が必要です（Colab には標準でインストール済み）
- 初回実行時は数分かかることがあります

In [ ]:
# gfortran がインストールされているか確認
!which gfortran
!gfortran --version

In [ ]:
# AI Feynman と依存パッケージをインストール
# sympy のバージョンを固定（互換性のため）
!pip install numpy
!pip install sympy==1.4
!pip install aifeynman

print("\n" + "="*50)
print("インストール完了!")
print("="*50)

### ランタイム再起動の確認

上記のインストール後、以下のセルを実行してください。エラーが出る場合は、メニューから「ランタイム」→「ランタイムを再起動」を選択してから再実行してください。

In [ ]:
# インストール確認
import aifeynman
print("AI Feynman が正常にインポートされました!")

---

## 2. 必要なライブラリのインポート

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# AI Feynman のメイン関数をインポート
from aifeynman import run_aifeynman, get_demos

print("ライブラリのインポート完了!")

---

## 3. サンプルデータのダウンロード

AI Feynman には、物理法則に基づいたサンプルデータセットが用意されています。

### サンプルデータの内容

| ファイル | 対応する物理法則 | 説明 |
|----------|------------------|------|
| example1.txt | Feynman I.8.14 | 距離の公式 |
| example2.txt | Feynman I.10.7 | 運動エネルギー関連 |
| example3.txt | Feynman I.50.26 | 波動関連 |

In [ ]:
# サンプルデータをダウンロード
# MIT のサーバーからデータを取得します
get_demos("example_data")

print("\nダウンロードしたファイル:")
!ls -la example_data/

---

## 4. データの確認と可視化

記号回帰を実行する前に、データの構造を確認しましょう。

In [ ]:
# example1.txt の最初の数行を確認
data = np.loadtxt("example_data/example1.txt")

print("データの形状:", data.shape)
print(f"サンプル数: {data.shape[0]}")
print(f"変数の数: {data.shape[1] - 1} (入力) + 1 (出力)")
print("\n最初の5行:")
print(data[:5])

In [ ]:
# データの可視化
# 注意: 軸ラベルは英語で表示（文字化け防止）

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 各入力変数と出力の関係をプロット
n_inputs = data.shape[1] - 1
output = data[:, -1]

for i in range(min(3, n_inputs)):
    ax = axes[i]
    # ランダムサンプリング（大量データの場合）
    sample_idx = np.random.choice(len(data), min(1000, len(data)), replace=False)
    ax.scatter(data[sample_idx, i], output[sample_idx], alpha=0.5, s=10)
    ax.set_xlabel(f'Input Variable x{i+1}')
    ax.set_ylabel('Output y')
    ax.set_title(f'x{i+1} vs Output')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. AI Feynman の実行

いよいよ記号回帰を実行します！

### パラメータの説明

| パラメータ | 説明 | 推奨値 |
|------------|------|--------|
| `pathdir` | データファイルのディレクトリパス | - |
| `filename` | データファイル名 | - |
| `BF_try_time` | 総当たり探索の制限時間（秒） | 30-60 |
| `BF_ops_file_type` | 使用する演算子セット | "14ops.txt" |
| `polyfit_deg` | 多項式フィッティングの最大次数 | 3-4 |
| `NN_epochs` | ニューラルネットワークの学習エポック数 | 500-4000 |

### 演算子セットの種類

- `7ops.txt`: 基本演算子（+, *, 等）
- `14ops.txt`: 三角関数・指数関数を含む（推奨）
- `19ops.txt`: 拡張演算子セット

### 実行時間の目安

- 簡単な問題: 5-10分
- 中程度の問題: 10-30分
- 複雑な問題: 30分以上

**注意**: GPU を使用すると高速化できます（Colab のメニューから「ランタイム」→「ランタイムのタイプを変更」→「GPU」を選択）

In [ ]:
# 結果を保存するディレクトリを作成
os.makedirs("results", exist_ok=True)

print("準備完了! 次のセルで AI Feynman を実行します。")

In [ ]:
%%time
# AI Feynman の実行
# 注意: この処理には数分〜数十分かかります

print("="*60)
print("AI Feynman 2.0 を実行中...")
print("="*60)
print("\n処理内容:")
print("  1. 総当たり探索（Brute-force search）")
print("  2. 多項式フィッティング（Polynomial fitting）")
print("  3. ニューラルネットワーク訓練（NN training）")
print("  4. 対称性・分離可能性の検出（Symmetry detection）")
print("  5. パレート最適化（Pareto optimization）")
print("\n" + "-"*60 + "\n")

# パラメータ設定（デモ用に軽量化）
run_aifeynman(
    pathdir="./example_data/",      # データディレクトリ
    filename="example1.txt",         # データファイル
    BF_try_time=30,                  # 総当たり探索の時間（秒）
    BF_ops_file_type="14ops.txt",    # 演算子セット
    polyfit_deg=3,                   # 多項式の最大次数
    NN_epochs=500                    # NNの学習エポック数
)

print("\n" + "="*60)
print("AI Feynman の実行が完了しました!")
print("="*60)

---

## 6. 結果の確認と解析

AI Feynman は、**パレートフロンティア**として結果を出力します。これは、複雑さと精度のトレードオフを示す複数の候補数式です。

### 出力フォーマット

結果ファイルの各行には以下の情報が含まれます：

1. **誤差（bit平均）**: 発見した数式の平均誤差（log2スケール）
2. **誤差（累積）**: 累積誤差
3. **複雑さ（bit）**: 数式の複雑さ（記述長）
4. **誤差（数値）**: 実際の誤差値
5. **発見した数式**: 記号式

In [ ]:
# 結果ファイルの確認
result_file = "results/solution_example1.txt"

if os.path.exists(result_file):
    print("発見された数式（パレートフロンティア）:")
    print("="*80)
    
    with open(result_file, 'r') as f:
        lines = f.readlines()
    
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) >= 4:
            # 最後の要素が数式
            equation = parts[-1]
            complexity = parts[-3] if len(parts) >= 4 else "N/A"
            error = parts[-2] if len(parts) >= 3 else "N/A"
            
            print(f"\n候補 {i+1}:")
            print(f"  数式: {equation}")
            print(f"  複雑さ: {complexity}")
            print(f"  誤差: {error}")
    
    print("\n" + "="*80)
else:
    print("結果ファイルが見つかりません。AI Feynman が正常に実行されたか確認してください。")
    print("\n利用可能なファイル:")
    !ls -la results/

In [ ]:
# 結果の可視化: パレートフロンティアのプロット

def parse_results(filepath):
    """結果ファイルをパースしてデータフレームに変換"""
    results = []
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 4:
                    try:
                        complexity = float(parts[-3])
                        error = float(parts[-2])
                        equation = parts[-1]
                        results.append({
                            'complexity': complexity,
                            'error': error,
                            'equation': equation
                        })
                    except (ValueError, IndexError):
                        continue
    return pd.DataFrame(results)

# 結果をパース
df_results = parse_results(result_file)

if len(df_results) > 0:
    # パレートフロンティアのプロット
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.scatter(df_results['complexity'], df_results['error'], 
               s=100, c='steelblue', alpha=0.7, edgecolors='navy')
    
    # パレートフロンティアを線で結ぶ
    df_sorted = df_results.sort_values('complexity')
    ax.plot(df_sorted['complexity'], df_sorted['error'], 
            'r--', alpha=0.5, linewidth=2, label='Pareto Frontier')
    
    ax.set_xlabel('Complexity (bits)', fontsize=12)
    ax.set_ylabel('Error', fontsize=12)
    ax.set_title('Pareto Frontier: Complexity vs Error Trade-off', fontsize=14)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # 最良の結果を表示
    print("\n最も精度が高い数式（誤差最小）:")
    best_idx = df_results['error'].idxmin()
    print(f"  {df_results.loc[best_idx, 'equation']}")
    print(f"  誤差: {df_results.loc[best_idx, 'error']:.6e}")
    
    print("\n最もシンプルな数式（複雑さ最小）:")
    simple_idx = df_results['complexity'].idxmin()
    print(f"  {df_results.loc[simple_idx, 'equation']}")
    print(f"  複雑さ: {df_results.loc[simple_idx, 'complexity']:.1f} bits")
else:
    print("結果データがありません。")

---

## 7. 発見した数式の検証

発見された数式が実際のデータにどれだけフィットするか確認します。

In [ ]:
from sympy import symbols, sympify, lambdify
from sympy.parsing.sympy_parser import parse_expr

def evaluate_equation(equation_str, data):
    """発見された数式を評価する"""
    n_vars = data.shape[1] - 1
    
    # 変数を作成
    var_names = [f'x{i}' for i in range(n_vars)]
    sym_vars = symbols(' '.join(var_names))
    if n_vars == 1:
        sym_vars = [sym_vars]
    
    try:
        # 数式をパース
        expr = parse_expr(equation_str)
        
        # 数値関数に変換
        f = lambdify(sym_vars, expr, modules=['numpy'])
        
        # 予測値を計算
        inputs = [data[:, i] for i in range(n_vars)]
        predictions = f(*inputs)
        
        return predictions
    except Exception as e:
        print(f"数式の評価中にエラー: {e}")
        return None

# 最良の数式で予測
if len(df_results) > 0:
    best_equation = df_results.loc[df_results['error'].idxmin(), 'equation']
    print(f"検証する数式: {best_equation}")
    
    predictions = evaluate_equation(best_equation, data)
    
    if predictions is not None:
        actual = data[:, -1]
        
        # サンプリング
        sample_size = min(1000, len(data))
        sample_idx = np.random.choice(len(data), sample_size, replace=False)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 予測 vs 実測プロット
        ax1 = axes[0]
        ax1.scatter(actual[sample_idx], predictions[sample_idx], alpha=0.5, s=10)
        
        # 対角線（完全一致の場合）
        min_val = min(actual.min(), np.nanmin(predictions))
        max_val = max(actual.max(), np.nanmax(predictions))
        ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Fit')
        
        ax1.set_xlabel('Actual Values', fontsize=12)
        ax1.set_ylabel('Predicted Values', fontsize=12)
        ax1.set_title('Predicted vs Actual', fontsize=14)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 残差プロット
        ax2 = axes[1]
        residuals = predictions[sample_idx] - actual[sample_idx]
        ax2.scatter(actual[sample_idx], residuals, alpha=0.5, s=10)
        ax2.axhline(y=0, color='r', linestyle='--', linewidth=2)
        
        ax2.set_xlabel('Actual Values', fontsize=12)
        ax2.set_ylabel('Residuals (Predicted - Actual)', fontsize=12)
        ax2.set_title('Residual Plot', fontsize=14)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # 統計情報
        mse = np.nanmean((predictions - actual)**2)
        mae = np.nanmean(np.abs(predictions - actual))
        print(f"\n統計情報:")
        print(f"  平均二乗誤差 (MSE): {mse:.6e}")
        print(f"  平均絶対誤差 (MAE): {mae:.6e}")

---

## 8. カスタムデータでの実行

自分のデータで AI Feynman を実行する方法を説明します。

### データ形式の要件

- **テキストファイル**（.txt）形式
- 各列はスペース、カンマ、またはタブで区切る
- 最初の n-1 列が**入力変数**、最後の列が**出力変数**
- ヘッダー行なし（数値データのみ）

### データ形式の例

```
1.0 2.0 3.0    <- x1=1.0, x2=2.0, y=3.0
2.0 3.0 5.0    <- x1=2.0, x2=3.0, y=5.0
3.0 4.0 7.0    <- x1=3.0, x2=4.0, y=7.0
...
```

In [ ]:
# カスタムデータの作成例
# ここでは y = x1^2 + x2 という関係を持つデータを生成

np.random.seed(42)

# データ生成
n_samples = 10000
x1 = np.random.uniform(0.1, 5.0, n_samples)
x2 = np.random.uniform(0.1, 5.0, n_samples)

# 真の関係式: y = x1^2 + x2
y_true = x1**2 + x2

# 少量のノイズを追加
noise = np.random.normal(0, 0.01, n_samples)
y = y_true + noise

# データを結合
custom_data = np.column_stack([x1, x2, y])

# カスタムデータディレクトリを作成
os.makedirs("custom_data", exist_ok=True)

# データを保存
np.savetxt("custom_data/my_equation.txt", custom_data)

print("カスタムデータを作成しました!")
print(f"真の関係式: y = x1^2 + x2")
print(f"データ形状: {custom_data.shape}")
print(f"\n最初の5行:")
print(custom_data[:5])

In [ ]:
%%time
# カスタムデータで AI Feynman を実行
# 注意: この処理には数分かかります

print("="*60)
print("カスタムデータで AI Feynman を実行中...")
print("真の関係式: y = x1^2 + x2")
print("="*60 + "\n")

run_aifeynman(
    pathdir="./custom_data/",
    filename="my_equation.txt",
    BF_try_time=30,
    BF_ops_file_type="14ops.txt",
    polyfit_deg=3,
    NN_epochs=500
)

print("\n" + "="*60)
print("完了!")
print("="*60)

In [ ]:
# カスタムデータの結果を確認
custom_result_file = "results/solution_my_equation.txt"

if os.path.exists(custom_result_file):
    print("発見された数式:")
    print("="*60)
    
    with open(custom_result_file, 'r') as f:
        lines = f.readlines()
    
    for i, line in enumerate(lines[-5:]):  # 最後の5つを表示
        parts = line.strip().split()
        if len(parts) >= 1:
            equation = parts[-1]
            print(f"  {equation}")
    
    print("\n" + "="*60)
    print("真の関係式: y = x1^2 + x2")
    print("="*60)
else:
    print("結果ファイルが見つかりません。")
    !ls -la results/

---

## 9. Google Drive へのデータ保存（オプション）

結果を Google Drive に保存して、後で利用できるようにします。

In [ ]:
# Google Drive をマウント（オプション）
# 実行すると認証画面が表示されます

from google.colab import drive
drive.mount('/content/drive')

# 結果をGoogle Driveにコピー
import shutil

save_path = "/content/drive/MyDrive/AI_Feynman_Results"
os.makedirs(save_path, exist_ok=True)

# results フォルダをコピー
if os.path.exists("results"):
    for file in os.listdir("results"):
        if file.startswith("solution_"):
            shutil.copy(f"results/{file}", f"{save_path}/{file}")
            print(f"保存完了: {file}")

print(f"\n結果を保存しました: {save_path}")

---

## 10. トラブルシューティング

### よくある問題と解決策

| 問題 | 解決策 |
|------|--------|
| `ModuleNotFoundError` | ランタイムを再起動してから再実行 |
| メモリ不足 | データサイズを減らすか、GPU ランタイムを使用 |
| 実行が遅い | `BF_try_time` や `NN_epochs` を減らす |
| 数式が見つからない | パラメータを調整（探索時間を増やす等） |
| Fortran エラー | gfortran がインストールされているか確認 |

### パラメータ調整のヒント

- **精度を上げたい場合**: `BF_try_time` を 60-120 に、`NN_epochs` を 2000-4000 に
- **速度を上げたい場合**: `BF_try_time` を 10-30 に、`NN_epochs` を 100-500 に
- **シンプルな数式を探したい場合**: `BF_ops_file_type="7ops.txt"` を使用

In [ ]:
# システム情報の確認
print("=" * 50)
print("システム情報")
print("=" * 50)

# GPU の確認
import torch
print(f"\nPyTorch バージョン: {torch.__version__}")
print(f"CUDA 利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU デバイス: {torch.cuda.get_device_name(0)}")

# メモリ情報
!free -h

# ディスク容量
print("\nディスク容量:")
!df -h /content

---

## まとめ

このノートブックでは、AI Feynman 2.0 を使って以下のことを学びました：

1. **環境セットアップ**: Google Colab での AI Feynman のインストール方法
2. **基本的な使い方**: サンプルデータでの記号回帰の実行
3. **結果の解析**: パレートフロンティアの理解と可視化
4. **数式の検証**: 発見された数式の精度確認
5. **カスタムデータ**: 自分のデータでの実行方法

### 次のステップ

- より複雑なデータセットで試してみる
- パラメータを調整して最適な設定を見つける
- 次元解析機能を使って物理単位を考慮した探索を行う

### 参考リンク

- [AI Feynman GitHub](https://github.com/SJ001/AI-Feynman)
- [Feynman Symbolic Regression Database](https://space.mit.edu/home/tegmark/aifeynman.html)
- [論文 (Science Advances)](https://advances.sciencemag.org/content/6/16/eaay2631)

---

**作成者**: AI Feynman 2.0 チュートリアル  
**最終更新**: 2024年